# OCTA Image Quality Assessment

In [ ]:
import torch
import os
import shutil
from pathlib import Path
from scripts.functions import (
    train_and_evaluate, classify_all_images,
    load_trained_model, evaluate_on_test_set,
    create_data_loaders, FocalLoss
)
from PIL import Image
import matplotlib.pyplot as plt



BASE_DIR        = Path(os.getcwd())
PREPROCESSING   = BASE_DIR.parent / "01_Data_Preprocessing"

DATA_ALL    = PREPROCESSING / "data_all"
MASTER_CSV  = PREPROCESSING / "master_dataset.csv"

LABELS_CSV  = BASE_DIR / "data" / "labeled_dataset.csv"
SPLIT_DIR   = BASE_DIR / "data" / "data_split_quality"
OUTPUT_FILE = BASE_DIR / "results" / "quality_predictions.xlsx"

TARGET_DATASETS = ['Drac', 'M3OCTA', 'Soul']

CONFIG = {
    'model_name':    'convnext_tiny',
    'iteration':     28,
    'batch_size':    16,
    'learning_rate': 0.00005,
    'num_epochs':    120,
    'img_size':      224,
    'dropout_rate':  0.25,
    'weight_decay':  0.005,
    'patience':      30,
    'num_workers':   4,
    'device':        torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

MODEL_PATH = os.path.join(
    'results',
    'cnn_model',
    f"{CONFIG['model_name']}_{CONFIG['iteration']}",
    'model.pth'
)
# -----------------------------------------------------------------------
print(f'Device:       {CONFIG["device"]}')
print(f'Model path:   {MODEL_PATH}')
print(f'Model exists: {os.path.exists(MODEL_PATH)}')

## Tréning modelu

In [ ]:
if os.path.exists(MODEL_PATH):
    print('Model už bol natrénovaný.')

    model, class_names, checkpoint = load_trained_model(
        MODEL_PATH,
        CONFIG['device']
    )

    print(f'\nModel: {checkpoint["model_name"]}')
    print(f'Najlepšia validačná presnosť: {checkpoint["val_acc"]:.2f}%')
    print(f'Triedy: {checkpoint["class_names"]}')

    output_dir = os.path.dirname(MODEL_PATH)

    report_path = os.path.join(output_dir, 'test_classification_report.txt')
    cm_path = os.path.join(output_dir, 'test_confusion_matrix.png')

    # =========================================================
    # TESTOVANIE UŽ EXISTUJE
    # =========================================================
    if os.path.exists(report_path) and os.path.exists(cm_path):

        print('\nVýsledky testovania už existujú.')
        print('\nNačítavam uložené výsledky testovania...\n')

        # VYPÍSANIE REPORTU
        with open(report_path, 'r') as f:
            print(f.read())

        # ZOBRAZENIE CONFUSION MATRIX
        img = Image.open(cm_path)

        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Uložená confusion matrix - testovacia množina')
        plt.show()

    # =========================================================
    # TESTOVANIE NEEXISTUJE
    # =========================================================
    else:
        print('\nUložené výsledky testovania neboli nájdené.')
        print('Spúšťam vyhodnotenie na testovacej množine...\n')

        if os.path.exists(os.path.join(SPLIT_DIR, 'test_split.csv')):

            _, val_loader, test_loader, _, _, test_dataset = create_data_loaders(
                SPLIT_DIR,
                DATA_ALL,
                CONFIG['batch_size'],
                CONFIG['img_size'],
                CONFIG['num_workers']
            )

            import torch

            class_counts = torch.zeros(len(class_names))

            for _, label in test_dataset:
                class_counts[label] += 1

            class_weights = 1.0 / class_counts
            class_weights = class_weights / class_weights.sum()
            class_weights = class_weights.to(CONFIG['device'])

            criterion = FocalLoss(alpha=class_weights, gamma=2)

            evaluate_on_test_set(
                model,
                test_loader,
                criterion,
                class_names,
                CONFIG['device'],
                output_dir
            )

        else:
            print('Testovací split nebol nájdený. Najskôr spusti tréning.')

# =============================================================
# TRÉNOVANIE MODELU
# =============================================================
else:
    print('Nenašiel sa natrénovaný model. Spúšťam tréning...')

    shutil.rmtree(SPLIT_DIR, ignore_errors=True)

    output_dir, best_val_acc, test_acc = train_and_evaluate(
        CONFIG,
        DATA_ALL,
        LABELS_CSV,
        SPLIT_DIR
    )

    MODEL_PATH = os.path.join(output_dir, 'model.pth')

    print(f'\nModel bol uložený do: {MODEL_PATH}')

## Klasikiácia všetkých snímkov

In [ ]:
df_results = classify_all_images(
    model_path      = MODEL_PATH,
    image_folder    = DATA_ALL,
    master_csv      = MASTER_CSV,
    target_datasets = TARGET_DATASETS,
    output_file     = OUTPUT_FILE,
    device          = CONFIG['device']
)

print(f'\nResults saved to: {OUTPUT_FILE}')